# Index Faiss : tests de recherche et choix de l'algorithme

**Objectifs**
- **A.** Vérifier la qualité de la recherche sémantique sur l'index complet (Nouvelle-Aquitaine, photo du 29/10/2026), avec des questions réalistes et une question hors sujet.
- **B.** Comparer, sur nos vrais vecteurs, l'index exact **Flat** aux index approchés **IVF** et **HNSW** : temps de construction, mémoire, temps de recherche et **rappel@5**.
- **C.** Situer le temps de la recherche dans le traitement d'une question : il est comparé au temps de vectorisation de la question par Mistral.

**Consignes** (énoncé, étape 3) : « Faites des **tests de recherche** pour vérifier l'efficacité » ; « S'assurer que l'index est optimisé pour des recherches rapides (**utiliser les bons algorithmes** dans Faiss) ».

**Prérequis** : l'index de référence, construit avec `python -m scripts.build_index` (≈ 12 min) ou téléchargé par `python -m scripts.serve` (index publié, empreinte vérifiée ; arrêter ensuite l'API avec Ctrl+C), et la clé `MISTRAL_API_KEY` dans `.env`. La partie A vectorise 6 questions et la partie C 3 questions (coût négligeable) ; la partie B n'appelle pas Mistral.

**Relance** : la cellule de chargement télécharge le tokenizer de Mistral depuis Hugging Face ; les parties A et C appellent Mistral. Si Mistral refuse (erreur 401, 402 ou 429), « Run All » s'arrête dans la partie A et n'atteint pas la partie B, qui peut alors être lancée cellule par cellule après le chargement. Les sorties enregistrées font foi.

**Exécution** : noyau `.venv` (Python 3.12), puis « Run All ».

## Chargement de l'index

In [2]:
import sys
import time
from pathlib import Path

# Le notebook s'exécute depuis notebooks/ : on ajoute la racine du projet au chemin d'import.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from rag import config
from rag.indexing import load_manifest, load_vectorstore, vectorstore_dir_for

load_dotenv(PROJECT_ROOT / ".env")  # charge MISTRAL_API_KEY (jamais affichée)

INDEX_DIR = vectorstore_dir_for(config.REFERENCE_RAW_FILE)
store = load_vectorstore(INDEX_DIR)
manifest = load_manifest(INDEX_DIR)
print(f"Index {INDEX_DIR.name} : {manifest['chunks']} chunks, {manifest['events']} événements, "
      f"{manifest['index_type']}, dimension {manifest['dimension']}")

Index nouvelle-aquitaine_2026-10-29 : 20644 chunks, 18771 événements, IndexFlatIP, dimension 1024


## A. Tests de recherche

Pour chaque question, les 5 chunks les plus proches, avec leur **score** (cosinus : vecteurs normalisés, index à produit scalaire).

**Hypothèses** :
- les questions thématiques (balade, concert, enfants) devraient trouver des événements pertinents, avec des scores nettement au-dessus de ceux d'une question hors sujet ;
- les questions qui précisent une **ville** risquent de remonter des événements d'autres villes, car la recherche sémantique capte mal les noms de lieux ;
- la question hors sujet devrait donner les scores les plus faibles, mais renverra quand même 5 résultats.

In [3]:
QUESTIONS = [
    "Faire une balade dans la nature",
    "Un concert de jazz à Bordeaux",
    "Une activité pour des enfants",
    "Une exposition à Limoges",
    "Du théâtre à Poitiers",
    "Quelle est la capitale de l'Australie ?",   # hors sujet : aucun événement ne devrait correspondre
]


def search(question: str, k: int = 5) -> pd.DataFrame:
    """Les k chunks les plus proches de la question (score = cosinus)."""
    results = store.similarity_search_with_score(question, k=k)
    time.sleep(1.1)  # la question est vectorisée par Mistral : 1 requête par seconde au maximum
    return pd.DataFrame([{
        "score": round(float(score), 3),
        "titre": doc.metadata["title"][:60],
        "ville": doc.metadata["city"],
        "début": doc.metadata["start"][:10],
        "événement": doc.metadata["uid"],
    } for doc, score in results])


results = {}
for question in QUESTIONS:
    results[question] = search(question)
    print(f"\n« {question} »")
    display(results[question])


« Faire une balade dans la nature »


,score,titre,ville,début,événement
0,0.829,Balade nature ludique à Bel-Air,Le Haillan,2026-04-22,53341738
1,0.825,Balades contées sur la nature,La Rochelle,2026-09-19,28576489
2,0.823,Balade nature - À la découverte des arbres,Artigues-près-Bordeaux,2025-12-06,88941292
3,0.820,Balade en forêt,Bègles,2025-12-06,5852785
4,0.820,"Escapade d'automne, balade botanique",Le Taillan-Médoc,2026-11-14,52912229



« Un concert de jazz à Bordeaux »


,score,titre,ville,début,événement
0,0.870,Concert Jazz & Co,Bordeaux,2026-11-25,97173163
1,0.857,Petit concert de Jazz,Bordeaux,2026-06-19,38962553
2,0.851,Concert du dimanche,Bordeaux,2027-01-03,84607589
3,0.843,Mardis de la musique : La France et le jazz,Bordeaux,2026-01-13,2708786
4,0.838,Concert Jazz-Band OHB,Bordeaux,2026-02-08,95683735



« Une activité pour des enfants »


,score,titre,ville,début,événement
0,0.776,Dessin pour les enfants,Bordeaux,2026-06-09,54703422
1,0.775,Activité ludique sur le thème,Bordeaux,2026-08-14,34154614
2,0.766,Collage - Souvenirs d’enfance / Tout public,Bordeaux,2026-03-27,22167086
3,0.764,L'œuf mystère,Bordeaux,2026-02-13,48995690
4,0.764,Ici on joue !,Bordeaux,2026-02-12,49951525


... (2 outputs left out)


« Du théâtre à Poitiers »


,score,titre,ville,début,événement
0,0.840,Visites techniques du TAP,Poitiers,2026-09-19,39694355
1,0.839,Visite du TAP,Poitiers,2026-09-19,88930307
2,0.827,Visite en famille du TAP,Poitiers,2026-09-19,78138163
3,0.827,LIEUX COMMUNS,Poitiers,2026-04-23,29726681
4,0.824,Exposition : « L’Odyssée du théâtre municipal ...,Châtellerault,2026-09-19,24359274



« Quelle est la capitale de l'Australie ? »


,score,titre,ville,début,événement
0,0.801,Vins d'Australie,Bordeaux,2026-07-30,24304870
1,0.796,Vins d'Australie,Bordeaux,2026-07-30,24304870
2,0.790,"Cap sur les vins d'Adelaide, Australie, et de ...",Bordeaux,2025-11-07,27023041
3,0.781,"Bouger à l'étranger, c'est possible ?! SPÉCIAL...",,2026-11-04,46137515
4,0.779,"Webinaire ""Partir en Australie & Nouvelle-Zéla...",,2026-09-29,69515437


In [4]:
# Synthèse par question : meilleur score, écart entre le 1er et le 5e résultat, villes et événements distincts
summary = pd.DataFrame([{
    "question": question,
    "meilleur score": table["score"].max(),
    "écart 1er - 5e": round(table["score"].iloc[0] - table["score"].iloc[-1], 3),
    "villes trouvées": ", ".join(sorted(set(table["ville"]))),
    "événements distincts (sur 5)": table["événement"].nunique(),
} for question, table in results.items()])
summary

,question,meilleur score,écart 1er - 5e,villes trouvées,événements distincts (sur 5)
0,Faire une balade dans la nature,0.829,0.009,"Artigues-près-Bordeaux, Bègles, La Rochelle, L...",5
1,Un concert de jazz à Bordeaux,0.870,0.032,Bordeaux,5
2,Une activité pour des enfants,0.776,0.012,Bordeaux,5
3,Une exposition à Limoges,0.857,0.009,Limoges,5
4,Du théâtre à Poitiers,0.840,0.016,"Châtellerault, Poitiers",5
5,Quelle est la capitale de l'Australie ?,0.801,0.022,", Bordeaux",4


## B. Comparatif Flat / IVF / HNSW

**Méthode**
- Les 20 644 vecteurs de l'index sont relus ; 200 d'entre eux, tirés au hasard, servent de requêtes (aucun appel à Mistral).
- Chaque requête étant elle-même dans l'index, on l'exclut de ses propres résultats.
- **Référence** : l'index **Flat**, exact. Le **rappel@5** d'un index est la part des 5 vrais plus proches voisins (ceux de Flat) qu'il retrouve.
- **IVF** : nombre de groupes fixé à 4√N, borne basse conseillée par le guide Faiss (« Guidelines to choose an index ») pour moins d'un million de vecteurs ; `nprobe` = nombre de groupes fouillés.
- **HNSW** : `M = 32` voisins par nœud ; `efSearch` = largeur de l'exploration.
- Temps de recherche : une question à la fois, comme dans l'API.

In [5]:
vectors = store.index.reconstruct_n(0, store.index.ntotal)   # tous les vecteurs (float32, normalisés)
n, d = vectors.shape
rng = np.random.default_rng(42)
query_ids = rng.choice(n, size=200, replace=False)
queries = vectors[query_ids]
K = 5


def neighbours(index, k: int = K) -> list[list[int]]:
    """Les k voisins de chaque requête, sans la requête elle-même (elle fait partie de l'index)."""
    _, ids = index.search(queries, k + 1)
    return [[int(i) for i in row if i != q][:k] for row, q in zip(ids, query_ids)]


def recall(found: list[list[int]], truth: list[list[int]]) -> float:
    """Part moyenne des vrais voisins (Flat) retrouvés."""
    return float(np.mean([len(set(f) & set(t)) / K for f, t in zip(found, truth)]))


def latency_ms(index) -> float:
    """Temps moyen d'une recherche, une question à la fois."""
    start = time.perf_counter()
    for query in queries:
        index.search(query.reshape(1, -1), K + 1)
    return (time.perf_counter() - start) / len(queries) * 1000


def memory_mb(index) -> float:
    """Taille de l'index sérialisé, en mégaoctets."""
    return faiss.serialize_index(index).nbytes / 1e6


rows = []


def measure(name: str, setting: str, index, build_seconds: float, truth: list[list[int]]) -> None:
    rows.append({
        "index": name,
        "réglage": setting,
        "construction (s)": round(build_seconds, 2),
        "mémoire (Mo)": round(memory_mb(index), 1),
        "recherche (ms/question)": round(latency_ms(index), 3),
        "rappel@5": round(recall(neighbours(index), truth), 3),
    })


# Flat : la référence exacte
start = time.perf_counter()
flat = faiss.IndexFlatIP(d)
flat.add(vectors)
flat_seconds = time.perf_counter() - start
truth = neighbours(flat)
measure("Flat", "exact", flat, flat_seconds, truth)
print(f"{n} vecteurs de dimension {d} ; {len(queries)} requêtes")

20644 vecteurs de dimension 1024 ; 200 requêtes


In [6]:
# IVF : partition en groupes (k-means), on ne fouille que nprobe groupes
nlist = int(4 * np.sqrt(n))
start = time.perf_counter()
quantizer = faiss.IndexFlatIP(d)
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(vectors)
ivf.add(vectors)
ivf_seconds = time.perf_counter() - start
for nprobe in [1, 8, 32]:
    ivf.nprobe = nprobe
    measure("IVF", f"{nlist} groupes, nprobe = {nprobe}", ivf, ivf_seconds, truth)

# HNSW : graphe de voisins
start = time.perf_counter()
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.add(vectors)
hnsw_seconds = time.perf_counter() - start
for ef in [16, 64, 128]:
    hnsw.hnsw.efSearch = ef
    measure("HNSW", f"M = 32, efSearch = {ef}", hnsw, hnsw_seconds, truth)

pd.DataFrame(rows)

WARNING clustering 20644 points to 574 centroids: please provide at least 22386 training points


,index,réglage,construction (s),mémoire (Mo),recherche (ms/question),rappel@5
0,Flat,exact,0.01,84.6,1.139,1.000
1,IVF,"574 groupes, nprobe = 1",0.32,87.1,0.040,0.640
2,IVF,"574 groupes, nprobe = 8",0.32,87.1,0.071,0.885
3,IVF,"574 groupes, nprobe = 32",0.32,87.1,0.163,0.947
4,HNSW,"M = 32, efSearch = 16",1.04,90.2,0.073,0.984
5,HNSW,"M = 32, efSearch = 64",1.04,90.2,0.189,0.997
6,HNSW,"M = 32, efSearch = 128",1.04,90.2,0.337,0.999


## C. Temps de vectorisation d'une question

La recherche dans Faiss n'est qu'une étape du traitement d'une question : celle-ci doit d'abord être vectorisée par Mistral (requête réseau). On mesure ce temps sur 3 questions, pour situer le temps de recherche de Flat.

In [7]:
latencies = []
for question in ["Un concert ce week-end", "Une sortie en famille", "Une exposition de photographie"]:
    start = time.perf_counter()
    store.embeddings.embed_query(question)          # requête à l'API Mistral (aller-retour réseau compris)
    latencies.append((time.perf_counter() - start) * 1000)
    time.sleep(1.1)                                  # 1 requête par seconde au maximum

flat_ms = rows[0]["recherche (ms/question)"]
print("Vectorisation d'une question :", ", ".join(f"{ms:.0f} ms" for ms in latencies),
      f"(médiane : {np.median(latencies):.0f} ms)")
print(f"Recherche Flat : {flat_ms} ms, soit {flat_ms / np.median(latencies):.1%} du temps de vectorisation")

Vectorisation d'une question : 359 ms, 255 ms, 218 ms (médiane : 255 ms)
Recherche Flat : 1.139 ms, soit 0.4% du temps de vectorisation


## Synthèse

### A. Tests de recherche

| Question | Constat | Conséquence |
|---|---|---|
| Balade dans la nature | 5 balades nature (0,820 à 0,829) | Pertinent |
| Concert de jazz à Bordeaux | 5 résultats à Bordeaux, dont 4 concerts de jazz (0,838 à 0,870) | Pertinent |
| Activité pour des enfants | Activités jeunesse, toutes à Bordeaux (0,764 à 0,776) | Pertinent, mais scores plus faibles ; Bordeaux, très représentée dans les données, domine |
| Exposition à Limoges | 5 expositions à Limoges (0,848 à 0,857) | Pertinent |
| Théâtre à Poitiers | 4 résultats à Poitiers, dont 3 visites du bâtiment du TAP plutôt que des spectacles, et 1 à Châtellerault | La ville et le sens précis sont imparfaitement captés : filtre sur la ville et tri par le LLM nécessaires |
| Capitale de l'Australie (hors sujet) | « Vins d'Australie » (0,801), « Partir en Australie » ; un même événement deux fois ; scores (0,779 à 0,801) supérieurs à ceux de la question « enfants » | Un seuil de score ne suffit pas à repérer une question hors sujet : le LLM doit juger la pertinence ; regrouper les résultats par événement |

Des événements terminés remontent pour la plupart des questions ; pour trois d'entre elles (enfants, Limoges, Poitiers), les 5 résultats sont tous terminés au 29/10/2026 : un filtre sur les dates est indispensable.

**Hypothèses** :
- thèmes : la pertinence est confirmée, mais pas l'écart de scores avec la question hors sujet (« enfants » : 0,764 à 0,776, sous la question hors sujet : 0,779 à 0,801) ;
- villes : en grande partie infirmée, 14 résultats sur 15 sont dans la bonne ville (seul « Poitiers » en renvoie un à Châtellerault) ;
- hors sujet : infirmée, la question hors sujet ne donne pas les scores les plus faibles ; elle renvoie bien 5 résultats.

### B. Choix de l'index

| Index | Rappel@5 | Recherche (ms/question) | Mémoire (Mo) |
|---|---|---|---|
| Flat | 1,000 | 1,14 | 84,6 |
| IVF (574 groupes ; nprobe 1 / 8 / 32) | 0,640 / 0,885 / 0,947 | 0,04 à 0,16 | 87,1 |
| HNSW (M = 32 ; efSearch 16 / 64 / 128) | 0,984 / 0,997 / 0,999 | 0,07 à 0,34 | 90,2 |

Faiss signale par ailleurs que 20 644 vecteurs ne suffisent pas pour entraîner correctement 574 groupes IVF (22 386 au minimum).

### C. Le temps de recherche dans le traitement d'une question

La vectorisation d'une question par Mistral prend **218 à 359 ms** (médiane **255 ms**). La recherche Flat (1,14 ms) en représente **0,4 %**, et une part encore plus faible du temps de réponse complet, qui comprend aussi la génération de la réponse par le LLM. Gagner environ 1 ms avec un index approché ne changerait rien pour l'utilisateur.

### Conclusion

**Flat est retenu** : c'est le seul index exact et le plus économe en mémoire, et son temps de recherche est négligeable à l'échelle d'une question. Ce temps croissant proportionnellement au nombre de vecteurs (≈ 55 ms pour un million), IVF ou HNSW deviendraient pertinents à partir de plusieurs millions de vecteurs.